# Atividade 8 — Limpeza e preparação dos dados

Projeto: **Spam Blocker** · Lucas da Silva Vargas

Complemento ao código `aula8.py`, com apoio do ChatGPT na organização, inspeção e documentação. Mantém as duas transformações do script original. Base: SMS Spam Collection (UCI). Objetivo: preparar mensagens para futura classificação entre legítimas (`ham`) e indesejadas (`spam`).

Execute todas as células na ordem. Dependência: `pandas`. O arquivo `sms_original.zip` deve estar ao lado do notebook; se estiver ausente, o código baixa a fonte original. Nenhum modelo é treinado nesta etapa.

In [1]:
from pathlib import Path
from urllib.request import urlopen
import zipfile
import io
import pandas as pd

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
origem = Path("sms_original.zip")
if origem.exists():
    conteudo = origem.read_bytes()
else:
    with urlopen(url, timeout=60) as resposta:
        conteudo = resposta.read()
    origem.write_bytes(conteudo)

# Mesmo carregamento empregado em aula8.py.
with zipfile.ZipFile(io.BytesIO(conteudo)) as arquivo:
    with arquivo.open("SMSSpamCollection") as dados:
        df = pd.read_csv(dados, sep="\t", header=None,
                         names=["label", "message"], encoding="utf-8")
df_original = df.copy(deep=True)
print("Base carregada:", df.shape)
print("Versão do pandas:", pd.__version__)


Base carregada: (5572, 2)
Versão do pandas: 2.2.3


## 1. Inspeção inicial

`label` é a variável alvo e `message` contém o texto. São verificadas dimensões, tipos, ausentes, duplicatas e distribuição das classes. A verificação de ausentes usa a definição de nulo do pandas.

In [2]:
print("Dimensões:", df.shape)
print("Tipos:\n", df.dtypes.to_string())
print("Ausentes por coluna:\n", df.isna().sum().to_string())
print("Duplicatas de linhas completas:", int(df.duplicated().sum()))
distribuicao_inicial = pd.DataFrame({
    "quantidade": df["label"].value_counts(),
    "percentual": df["label"].value_counts(normalize=True).mul(100).round(2)
})
print("Distribuição inicial:\n", distribuicao_inicial.to_string())
assert set(df["label"].unique()) == {"ham", "spam"}


Dimensões: (5572, 2)
Tipos:
 label      object
message    object
Ausentes por coluna:
 label      0
message    0
Duplicatas de linhas completas: 403
Distribuição inicial:
        quantidade  percentual
label                        
ham          4825       86.59
spam          747       13.41


## 2. Remoção de duplicatas

Removo linhas iguais em `label` e `message`, mantendo a primeira ocorrência. Isso evita dar peso adicional a cópias idênticas e reduz o risco de a mesma mensagem aparecer em treino e teste. Prefiro essa estratégia a manter as cópias, mas reconheço que a frequência real de mensagens e a proporção das classes podem ser alteradas. A deduplicação exata não detecta mensagens apenas semelhantes.

In [3]:
linhas_antes = len(df)
duplicatas = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
linhas_depois = len(df)
print("Antes:", linhas_antes)
print("Removidas:", duplicatas)
print("Depois:", linhas_depois)
print("Duplicatas restantes:", int(df.duplicated().sum()))
print("Mensagens textualmente idênticas restantes:", int(df["message"].duplicated().sum()))
print("Classes após deduplicação:\n", df["label"].value_counts().to_string())
assert linhas_antes - linhas_depois == duplicatas
assert not df.duplicated().any()


Antes: 5572
Removidas: 403
Depois: 5169
Duplicatas restantes: 0
Mensagens textualmente idênticas restantes: 0
Classes após deduplicação:
 label
ham     4516
spam     653


## 3. Codificação da variável alvo

Uso o mapa fixo `ham → 0` e `spam → 1`. É uma codificação binária de classes, sem hierarquia ou intensidade entre elas. Não é necessário criar duas colunas por one-hot encoding para esse alvo binário. O mapeamento não aprende parâmetros da amostra. A validação detecta categorias desconhecidas que poderiam virar valores nulos. Não aplico escala diretamente ao texto nem preencho ausentes, pois não foram encontrados nulos.

In [4]:
mapa = {"ham": 0, "spam": 1}
linhas_codificadas = len(df)
df["label"] = df["label"].map(mapa)
assert df["label"].notna().all(), "Categoria não reconhecida no mapeamento"
assert set(df["label"].unique()) == {0, 1}
print("Rótulos convertidos:", linhas_codificadas)
print("Tipo final do alvo:", df["label"].dtype)
print("Nulos após conversão:", int(df["label"].isna().sum()))
print("Distribuição final:\n", pd.DataFrame({
    "quantidade": df["label"].value_counts(),
    "percentual": df["label"].value_counts(normalize=True).mul(100).round(2)
}).to_string())


Rótulos convertidos: 5169
Tipo final do alvo: int64
Nulos após conversão: 0
Distribuição final:
        quantidade  percentual
label                        
0            4516       87.37
1             653       12.63


## 4. Registro de decisões e impacto

As contagens abaixo são calculadas durante a execução. Nenhuma coluna foi removida; o texto de `message` foi preservado.

In [5]:
decisoes = pd.DataFrame([
    {"transformacao": "Remoção de duplicatas exatas", "coluna_afetada": "label e message",
     "motivo": "Evitar peso adicional de cópias e repetição entre treino e teste",
     "impacto": f"{duplicatas} linhas removidas; {linhas_antes} -> {linhas_depois}",
     "risco": "Alterar frequências e proporções; não detectar textos semelhantes"},
    {"transformacao": "Codificação binária fixa", "coluna_afetada": "label",
     "motivo": "Representar as duas classes por 0 e 1 sem criar duas colunas",
     "impacto": f"{linhas_codificadas} valores convertidos em 1 coluna; nenhuma linha removida",
     "risco": "Categoria desconhecida virar nulo; mitigado por validação"}
])
print(decisoes.to_string(index=False))
decisoes.to_csv("registro_decisoes.csv", index=False)


               transformacao  coluna_afetada                                                           motivo                                                      impacto                                                             risco
Remoção de duplicatas exatas label e message Evitar peso adicional de cópias e repetição entre treino e teste                           403 linhas removidas; 5572 -> 5169 Alterar frequências e proporções; não detectar textos semelhantes
    Codificação binária fixa           label     Representar as duas classes por 0 e 1 sem criar duas colunas 5169 valores convertidos em 1 coluna; nenhuma linha removida         Categoria desconhecida virar nulo; mitigado por validação


## 5. Verificação de vazamento de dados

As duas transformações aplicadas não estimam mediana, média, desvio padrão, vocabulário ou outro parâmetro de ajuste. Não há `fit` de imputador, escalonador ou codificador aprendido. A divisão treino/teste ainda não foi realizada. A deduplicação ocorre antes dessa futura divisão, e o mapa das classes é definido manualmente.

Na próxima etapa, será necessário separar treino/teste com estratificação por `label` e ajustar a vetorização do texto (por exemplo, TF-IDF) somente no treino. O teste receberá apenas `transform`, com o vocabulário e os pesos aprendidos no treino. O mesmo vale para futura imputação ou escala. A ausência de duplicatas exatas não garante ausência de mensagens semelhantes entre os conjuntos.

## 6. Salvamento e conferência

O CSV tratado é um novo arquivo, separado do ZIP original. Sua leitura de volta confere dimensões, conteúdo e ausência de duplicatas. O arquivo original permanece preservado.

In [6]:
df.to_csv("dados_tratados.csv", index=False)
conferencia = pd.read_csv("dados_tratados.csv")
pd.testing.assert_frame_equal(df.reset_index(drop=True), conferencia, check_dtype=False)
assert conferencia.isna().sum().sum() == 0
assert not conferencia.duplicated().any()
print("Arquivo salvo: dados_tratados.csv")
print("Original:", df_original.shape, "| Tratado:", conferencia.shape)
print("Colunas removidas: 0")
print("Ausentes finais:", conferencia.isna().sum().to_dict())
print("Conferência de leitura: OK")


Arquivo salvo: dados_tratados.csv
Original: (5572, 2) | Tratado: (5169, 2)
Colunas removidas: 0
Ausentes finais: {'label': 0, 'message': 0}
Conferência de leitura: OK


## 7. Organização e continuidade

O script original, a fonte preservada, o notebook executado, o CSV tratado e o registro das decisões ficam na mesma pasta. A principal questão técnica é manter a avaliação futura independente do treinamento. A base é desbalanceada: após a limpeza, spam representa cerca de 12,63% das mensagens. Essa característica deve orientar a divisão e a avaliação futuras.

**Uso de IA:** ChatGPT apoiou a construção deste notebook complementar, sua execução de conferência e a documentação. O código-base fornecido foi `aula8.py`. O estudante deve revisar o material e conseguir explicar cada decisão antes da entrega.